# 04e Phase 2：立春年 / 年五行（`year_element`）交互 gate（day_group × year_element）

目标：把“时运/大环境”叙事翻译为可证伪的交互检验，并采用层级检验：
- 层 1（Confirmatory gate）：交互项 joint test → 跨指数 Fisher 合并 → 对 `stem/branch/ganzhi_day` 做 BH-FDR
- 层 2（Exploratory）：仅当 gate 通过才落盘 cell-level 输出

输入：
- `data/clean/market_ganzhi_{ts_code}.csv.gz`
- `docs/li_chun_year_mapping.csv`

输出（写入 `data/clean/robustness/`）：
- `interaction_joint_test_{ts_code}_{day_group}_ret_1d.csv`
- `interaction_joint_tests_ret_1d.csv`
- `meta_interaction_{day_group}_ret_1d.csv`
- `interaction_gate_summary_ret_1d.csv`
- `interaction_cell_effects_{ts_code}_{day_group}_ret_1d.csv`（仅 gate 通过）


In [1]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import build_design_matrices
from scipy import stats

warnings.filterwarnings('ignore')

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

# plots
import matplotlib.pyplot as plt
import seaborn as sns


def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean'
ROBUST_DIR = CLEAN_DIR / 'robustness'
ROBUST_DIR.mkdir(parents=True, exist_ok=True)

# =====================
# Config (edit as needed)
# =====================
TS_CODES = ['000300.SH', '000852.SH', '000001.SH', '399001.SZ']

# Control-variable models to run
GROUP_COLS = ['stem', 'branch']
RUN_GANZHI_DAY_MODELS = True

# Permutation test (raw series; global)
RANDOM_SEED = 20260213
N_PERM = 1000

# Permutation on controls-only residuals (more conservative)
N_PERM_RESID = 2000

# HAC / Newey-West (time-series dependence)
HAC_MAXLAGS = 5
HAC_MAXLAGS_SWEEP = [0, 1, 3, 5, 10]

# Block bootstrap (controls-only residuals)
N_BOOT = 1000
BOOT_BLOCK_LEN = 10

# Walk-forward
TRAIN_YEARS = 5
TRAIN_YEARS_SWEEP = [3, 5, 7]

STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')

# Phase 2: Li-Chun year element interaction (day_group × year_element)
RUN_PHASE2_INTERACTIONS = True
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_P_THRESHOLD = 0.10
PHASE2_GATE_Q = 0.10
PHASE2_GATE_MIN_SHARE = 0.75


def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)


def load_market_ganzhi(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}. Run notebooks 01-03 first.')
    df = pd.read_csv(path, compression='gzip', dtype={'trade_date': str})
    if df.empty:
        raise ValueError(f'Empty data: {path}')
    return df


def add_calendar_controls(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['trade_date'], format='%Y%m%d')
    out['date'] = dt
    out['weekday'] = dt.dt.weekday.astype(int)
    out['month'] = dt.dt.month.astype(int)
    out['year'] = dt.dt.year.astype(int)
    return out


def set_category_orders(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'stem' in out.columns:
        out['stem'] = pd.Categorical(out['stem'], categories=STEMS_ORDER, ordered=True)
    if 'branch' in out.columns:
        out['branch'] = pd.Categorical(out['branch'], categories=BRANCHES_ORDER, ordered=True)
    return out


def load_li_chun_year_mapping() -> pd.DataFrame:
    path = ROOT / 'docs/li_chun_year_mapping.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path} (required for year_element).')

    m = pd.read_csv(
        path,
        dtype={
            'li_chun_date_yyyymmdd': str,
            'li_chun_begin_local': str,
            'year_stem': str,
            'year_element': str,
        },
    )
    if m.empty:
        raise ValueError(f'Empty mapping: {path}')

    required = {'li_chun_year', 'li_chun_date_yyyymmdd', 'li_chun_begin_local', 'year_stem', 'year_element'}
    missing = required - set(m.columns)
    if missing:
        raise ValueError(f'li_chun_year_mapping.csv missing columns: {sorted(missing)}')

    m = m.copy()
    m['li_chun_year'] = m['li_chun_year'].astype(int)
    m['li_chun_date_yyyymmdd'] = (
        m['li_chun_date_yyyymmdd'].astype(str).str.replace('.0', '', regex=False).str.zfill(8)
    )
    m['li_chun_time_hm'] = m['li_chun_begin_local'].astype(str).str[-5:]
    m = m.drop_duplicates(subset=['li_chun_year']).sort_values(['li_chun_year']).reset_index(drop=True)
    return m


def add_li_chun_year_fields(df: pd.DataFrame, mapping: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Always recompute from mapping to ensure a frozen, auditable rule.

    li_chun_date_by_year = mapping.set_index('li_chun_year')['li_chun_date_yyyymmdd'].to_dict()
    li_chun_time_by_year = mapping.set_index('li_chun_year')['li_chun_time_hm'].to_dict()
    year_stem_by_year = mapping.set_index('li_chun_year')['year_stem'].to_dict()
    year_element_by_year = mapping.set_index('li_chun_year')['year_element'].to_dict()

    trade_year = out['trade_date'].str.slice(0, 4).astype(int)
    li_chun_date_for_year = trade_year.map(li_chun_date_by_year)
    li_chun_time_for_year = trade_year.map(li_chun_time_by_year)
    if li_chun_date_for_year.isna().any() or li_chun_time_for_year.isna().any():
        missing_years = sorted(
            trade_year[li_chun_date_for_year.isna() | li_chun_time_for_year.isna()].unique().tolist()
        )
        raise ValueError(
            f'Missing li_chun_date/time for trade_year(s) {missing_years} in docs/li_chun_year_mapping.csv'
        )

    trade_date = out['trade_date'].astype(str)
    is_before = trade_date < li_chun_date_for_year
    is_after = trade_date > li_chun_date_for_year
    is_same = ~(is_before | is_after)
    same_is_new = li_chun_time_for_year <= '15:00'
    use_prev = is_before | (is_same & (~same_is_new))
    li_chun_year = np.where(use_prev, trade_year - 1, trade_year)
    out['li_chun_year'] = li_chun_year.astype(int)

    out['year_stem'] = out['li_chun_year'].map(year_stem_by_year)
    out['year_element'] = out['li_chun_year'].map(year_element_by_year)
    if out['year_stem'].isna().any() or out['year_element'].isna().any():
        miss = sorted(out.loc[out['year_element'].isna() | out['year_stem'].isna(), 'li_chun_year'].unique().tolist())
        raise ValueError(f'Missing year mapping for li_chun_year(s) {miss} in docs/li_chun_year_mapping.csv')

    out['year_stem'] = out['year_stem'].astype(str)
    out['year_element'] = pd.Categorical(out['year_element'].astype(str), categories=YEAR_ELEMENT_ORDER, ordered=True)
    return out


def fit_phase2_interaction_model(df: pd.DataFrame, day_group: str):
    formula = f"ret_1d ~ C({day_group}) * C(year_element) + C(weekday) + C(month)"
    model = smf.ols(formula=formula, data=df)
    res = model.fit(cov_type='HAC', cov_kwds={'maxlags': HAC_MAXLAGS})
    return res, formula


def phase2_joint_test_interaction(res, day_group: str) -> dict:
    # Joint Wald test for all interaction terms: C(day_group):C(year_element)
    param_names = list(res.params.index)
    interaction_names = [
        n for n in param_names if (':' in n) and (f'C({day_group})' in n) and ('C(year_element)' in n)
    ]
    if not interaction_names:
        return {
            'n_interaction_params': 0,
            'df_interaction': 0,
            'wald_stat': np.nan,
            'p_interaction': np.nan,
        }

    idxs = [param_names.index(n) for n in interaction_names]
    R = np.zeros((len(idxs), len(param_names)), dtype=float)
    for r, i in enumerate(idxs):
        R[r, i] = 1.0

    wt = res.wald_test(R)
    stat = float(np.squeeze(np.asarray(wt.statistic)))
    pval = float(np.squeeze(np.asarray(wt.pvalue)))

    return {
        'n_interaction_params': int(len(idxs)),
        'df_interaction': int(len(idxs)),
        'wald_stat': stat,
        'p_interaction': pval,
    }


def fisher_combine_p(pvals: np.ndarray) -> tuple[float, float, int, int]:
    p = np.asarray(pvals, dtype=float)
    p = p[np.isfinite(p)]
    if p.size == 0:
        return (np.nan, np.nan, 0, 0)
    p = np.clip(p, 1e-300, 1.0)
    stat = float(-2.0 * np.sum(np.log(p)))
    df = int(2 * p.size)
    p_meta = float(stats.chi2.sf(stat, df))
    return (p_meta, stat, df, int(p.size))


def phase2_cell_marginal_means(res, df: pd.DataFrame, day_group: str) -> pd.DataFrame:
    # Marginal mean for each (day_group, year_element) combination, averaging over observed controls.
    if day_group not in df.columns:
        return pd.DataFrame()

    base_mean = float(np.nanmean(res.predict(df)))
    counts = (
        df.groupby([day_group, 'year_element'], dropna=False)
        .size()
        .rename('n_cell')
        .reset_index()
    )

    if isinstance(df[day_group].dtype, pd.CategoricalDtype):
        present = set(df[day_group].dropna())
        day_values = [v for v in df[day_group].dtype.categories if v in present]
    else:
        day_values = sorted(df[day_group].dropna().unique().tolist())

    if isinstance(df['year_element'].dtype, pd.CategoricalDtype):
        present = set(df['year_element'].dropna())
        year_values = [v for v in df['year_element'].dtype.categories if v in present]
    else:
        year_values = sorted(df['year_element'].dropna().unique().tolist())

    records = []
    for gv in day_values:
        for ye in year_values:
            df2 = df.copy()
            df2[day_group] = gv
            df2['year_element'] = ye
            mm = float(np.nanmean(res.predict(df2)))

            mask = (counts[day_group].astype(str) == str(gv)) & (counts['year_element'].astype(str) == str(ye))
            n_cell = int(counts.loc[mask, 'n_cell'].iloc[0]) if mask.any() else 0

            rec = {
                'group_value': str(gv),
                'year_element': str(ye),
                'n_cell': n_cell,
                'marginal_mean': mm,
                'marginal_mean_all': base_mean,
                'marginal_effect_vs_all': mm - base_mean,
            }

            if day_group == 'ganzhi_day' and 'jiazi_idx' in df.columns:
                vals = df.loc[df['ganzhi_day'].astype(str) == str(gv), 'jiazi_idx'].dropna().unique()
                if len(vals) == 1:
                    rec['jiazi_idx'] = int(vals[0])

            records.append(rec)

    out = pd.DataFrame.from_records(records)
    return out


PROJECT_ROOT = D:\Work\中国传统投资\风水五行阴阳天干地支


## 1) 交互 joint test + gate +（可选）cell-level 边际效应


In [2]:
# Phase 2: year_element interaction gate (day_group × year_element)
if RUN_PHASE2_INTERACTIONS:
    mapping = load_li_chun_year_mapping()

    fit_cache = {}
    joint_rows = []

    for ts_code in TS_CODES:
        df = load_market_ganzhi(ts_code)
        df = add_calendar_controls(df)
        df = set_category_orders(df)
        df = add_li_chun_year_fields(df, mapping)
        df = df.dropna(subset=['ret_1d', 'year_element', 'weekday', 'month'])
        df['year_element'] = pd.Categorical(df['year_element'], categories=YEAR_ELEMENT_ORDER, ordered=True)

        for day_group in PHASE2_DAY_GROUPS:
            if day_group not in df.columns:
                continue
            if not isinstance(df[day_group].dtype, pd.CategoricalDtype):
                df[day_group] = df[day_group].astype('category')

            try:
                res, formula = fit_phase2_interaction_model(df, day_group)
                jt = phase2_joint_test_interaction(res, day_group)
                row = {
                    'ts_code': ts_code,
                    'day_group': day_group,
                    'n_days': int(len(df)),
                    'wald_stat': float(jt.get('wald_stat', np.nan)),
                    'df_interaction': int(jt.get('df_interaction', 0)),
                    'p_interaction': float(jt.get('p_interaction', np.nan)),
                    'hac_maxlags': int(HAC_MAXLAGS),
                    'formula': formula,
                }
                fit_cache[(ts_code, day_group)] = {'df': df, 'res': res, 'formula': formula}
            except Exception as exc:
                row = {
                    'ts_code': ts_code,
                    'day_group': day_group,
                    'n_days': int(len(df)),
                    'wald_stat': np.nan,
                    'df_interaction': np.nan,
                    'p_interaction': np.nan,
                    'hac_maxlags': int(HAC_MAXLAGS),
                    'formula': f"ret_1d ~ C({day_group}) * C(year_element) + C(weekday) + C(month)",
                    'error': str(exc),
                }
                print(f'[{ts_code}] Phase2 interaction failed for {day_group}:', exc)

            joint_rows.append(row)
            out_path = ROBUST_DIR / f'interaction_joint_test_{ts_code}_{day_group}_ret_1d.csv'
            pd.DataFrame([row]).to_csv(out_path, index=False)
            print('saved:', out_path)

    joint_df = pd.DataFrame.from_records(joint_rows)
    joint_long_path = ROBUST_DIR / 'interaction_joint_tests_ret_1d.csv'
    joint_df.to_csv(joint_long_path, index=False)
    print('saved:', joint_long_path)

    meta_rows = []
    for day_group in PHASE2_DAY_GROUPS:
        sub = joint_df[(joint_df['day_group'] == day_group) & np.isfinite(joint_df['p_interaction'])].copy()
        pvals = sub['p_interaction'].to_numpy(dtype=float)
        p_meta, fisher_stat, fisher_df, k_used = fisher_combine_p(pvals)
        n_pass = int((pvals <= PHASE2_P_THRESHOLD).sum()) if k_used else 0
        min_required = int(np.ceil(PHASE2_GATE_MIN_SHARE * k_used)) if k_used else 0
        meta_rows.append(
            {
                'day_group': day_group,
                'k_indices_used': int(k_used),
                'n_pass_p_interaction': n_pass,
                'min_pass_required': min_required,
                'fisher_stat': fisher_stat,
                'fisher_df': int(fisher_df),
                'p_meta_interaction': p_meta,
            }
        )

    meta_df = pd.DataFrame.from_records(meta_rows)

    meta_df['q_meta_interaction'] = np.nan
    ok = np.isfinite(meta_df['p_meta_interaction'].to_numpy(dtype=float))
    if ok.sum() > 0:
        q = np.full(len(meta_df), np.nan, dtype=float)
        q[ok] = fdr_bh(meta_df.loc[ok, 'p_meta_interaction'].to_numpy(dtype=float))
        meta_df['q_meta_interaction'] = q

    meta_df['pass_q'] = np.isfinite(meta_df['q_meta_interaction']) & (meta_df['q_meta_interaction'] <= PHASE2_GATE_Q)
    meta_df['pass_p_count'] = meta_df['n_pass_p_interaction'] >= meta_df['min_pass_required']
    meta_df['pass_gate'] = meta_df['pass_q'] & meta_df['pass_p_count']

    p_wide = joint_df.pivot_table(index='day_group', columns='ts_code', values='p_interaction', aggfunc='first').reset_index()
    for c in list(p_wide.columns):
        if c != 'day_group':
            p_wide = p_wide.rename(columns={c: f'p_interaction_{c}'})

    gate_df = meta_df.merge(p_wide, on='day_group', how='left')
    gate_path = ROBUST_DIR / 'interaction_gate_summary_ret_1d.csv'
    gate_df.to_csv(gate_path, index=False)
    print('saved:', gate_path)

    for _, r in gate_df.iterrows():
        out_path = ROBUST_DIR / f"meta_interaction_{r['day_group']}_ret_1d.csv"
        pd.DataFrame([r]).to_csv(out_path, index=False)
        print('saved:', out_path)

    passed_groups = gate_df[gate_df['pass_gate'] == True]['day_group'].astype(str).tolist()
    if not passed_groups:
        print('Phase 2 gate not passed; skip cell-level outputs.')
    else:
        for (ts_code, day_group), item in fit_cache.items():
            if day_group not in passed_groups:
                continue
            df = item['df']
            res = item['res']
            formula = item['formula']
            cell = phase2_cell_marginal_means(res, df, day_group)
            if cell.empty:
                continue
            cell.insert(0, 'ts_code', ts_code)
            cell.insert(1, 'day_group', day_group)
            cell['hac_maxlags'] = int(HAC_MAXLAGS)
            cell['formula'] = formula
            out_path = ROBUST_DIR / f'interaction_cell_effects_{ts_code}_{day_group}_ret_1d.csv'
            cell.to_csv(out_path, index=False)
            print('saved:', out_path, '| rows=', len(cell))
else:
    print('RUN_PHASE2_INTERACTIONS=False; skip Phase 2 interaction tests.')




saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000300.SH_stem_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000300.SH_branch_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000300.SH_ganzhi_day_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000852.SH_stem_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000852.SH_branch_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000852.SH_ganzhi_day_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000001.SH_stem_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000001.SH_branch_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\clean\robustness\interaction_joint_test_000001.SH_ganzhi_day_ret_1d.csv
saved: D:\Work\中国传统投资\风水五行阴阳天干地支\data\cl